In [3]:
import pandas as pd
import numpy as np
from datetime import timedelta
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import entropy

In [11]:
smn_historico = pd.read_csv('../datasets/smn/smn_historico.csv', na_values=['S/D'])

In [12]:
smn_historico.columns

Index(['estacion', 'provincia', 'altura', 'numero', 'NroOACI', 'LAT_decimal',
       'LON_decimal', 'mes_txt', 'prec_sup_1mm', 'humedad', 'nubosidad',
       'prec_mm', 'temp', 'temp_max', 'temp_min', 'viento'],
      dtype='object')

In [16]:
provincias = ['MENDOZA', 'SAN JUAN', 'SAN LUIS', 'LA RIOJA']

smn_cuyo = smn_historico[smn_historico['provincia'].isin(provincias)]


In [19]:
smn_cuyo["estacion"].unique()

array(['CHAMICAL AERO', 'CHILECITO AERO', 'JACHAL', 'LA RIOJA AERO',
       'MALARGUE AERO', 'MENDOZA AERO', 'MENDOZA OBSERVATORIO',
       'SAN JUAN AERO', 'SAN LUIS AERO', 'SAN MARTIN (MZA)',
       'SAN RAFAEL AERO', 'VILLA REYNOLDS AERO'], dtype=object)

In [24]:
smn_cuyo["temp"].mean()

17.063194444444445

In [25]:
smn_cuyo["temp"].std()

6.295994359028603

In [26]:
smn_cuyo.head()

,estacion,provincia,altura,numero,NroOACI,LAT_decimal,LON_decimal,mes_txt,prec_sup_1mm,humedad,nubosidad,prec_mm,temp,temp_max,temp_min,viento
180,CHAMICAL AERO,LA RIOJA,461.0,87320.0,SACT,-30.333333,-66.283333,Ene,5.8,58.3,3.5,93.4,27.0,33.8,20.6,8.4
181,CHAMICAL AERO,LA RIOJA,461.0,87320.0,SACT,-30.333333,-66.283333,Feb,5.3,62.6,3.7,88.3,25.3,31.9,19.5,6.9
182,CHAMICAL AERO,LA RIOJA,461.0,87320.0,SACT,-30.333333,-66.283333,Mar,5.2,67.3,3.7,67.2,23.2,29.5,18.0,6.2
183,CHAMICAL AERO,LA RIOJA,461.0,87320.0,SACT,-30.333333,-66.283333,Abr,3.1,69.0,3.8,27.6,19.3,25.6,14.3,5.0
184,CHAMICAL AERO,LA RIOJA,461.0,87320.0,SACT,-30.333333,-66.283333,May,1.9,70.3,3.9,10.2,15.2,21.4,10.2,4.3


In [27]:
smn_cuyo.groupby("mes_txt")["prec_mm"].mean().sort_values(ascending=False)

mes_txt
Ene    59.045455
Feb    55.775000
Dic    48.690909
Mar    48.227273
Nov    37.063636
Abr    26.241667
Oct    23.325000
May    14.158333
Sep    13.016667
Ago     9.081818
Jun     7.200000
Jul     6.850000
Name: prec_mm, dtype: float64

In [28]:
smn_cuyo.groupby("provincia")["temp"].mean().sort_values(ascending=False)

provincia
LA RIOJA    19.577778
SAN JUAN    17.395833
SAN LUIS    16.650000
MENDOZA     15.586667
Name: temp, dtype: float64

In [29]:
def tiene_outliers(serie):
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1
    
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR
    
    return ((serie < limite_inf) | (serie > limite_sup)).any()

print("Temperatura:", tiene_outliers(smn_cuyo["temp"]))
print("Precipitación:", tiene_outliers(smn_cuyo["prec_mm"]))

Temperatura: False
Precipitación: True


In [30]:
smn_cuyo["altura"].corr(smn_cuyo["temp"], method="spearman")

-0.20924501712951257

In [31]:
media_total = smn_cuyo["prec_mm"].mean()

ss_entre = sum(
    len(grupo) * (grupo["prec_mm"].mean() - media_total)**2
    for _, grupo in smn_cuyo.groupby("provincia")
)

ss_total = ((smn_cuyo["prec_mm"] - media_total)**2).sum()

eta2 = ss_entre / ss_total

eta2

0.2804070149035949

In [32]:
provincia_elegida = "MENDOZA"

datos_provincia = smn_cuyo[smn_cuyo["provincia"] == provincia_elegida]

resumen = datos_provincia[["temp", "prec_mm", "humedad", "nubosidad", "viento"]].describe()

resumen

,temp,prec_mm,humedad,nubosidad,viento
count,60.000000,60.000000,60.000000,60.000000,24.000000
mean,15.586667,24.185000,56.086667,2.993333,10.504167
std,6.288932,13.560421,8.102447,0.397393,1.237276
min,3.800000,3.100000,40.000000,2.300000,8.200000
25%,10.325000,12.025000,49.925000,2.700000,9.625000
50%,15.950000,23.850000,54.100000,2.950000,10.400000
75%,21.050000,32.675000,63.275000,3.200000,11.450000
max,25.900000,54.200000,70.400000,4.000000,12.600000


In [35]:
datos_provincia.groupby("mes_txt")[["temp", "temp_max", "temp_min", "prec_mm", "humedad", "nubosidad", "viento"]].mean()

,temp,temp_max,temp_min,prec_mm,humedad,nubosidad,viento
mes_txt,,,,,,,
Abr,14.80,22.30,9.12,23.66,65.02,3.08,8.40
Ago,9.24,17.48,2.90,13.06,54.18,2.80,11.90
Dic,22.96,30.48,15.28,29.04,46.14,2.62,11.40
Ene,23.98,31.46,16.66,42.34,50.00,2.86,10.65
Feb,22.22,29.76,15.40,42.62,55.70,2.92,9.90
Jul,6.70,14.60,1.06,10.38,61.42,3.02,10.80
Jun,7.54,15.20,2.14,11.52,65.20,3.26,10.85
Mar,19.68,27.10,13.50,36.96,60.86,2.92,9.10
May,10.66,17.94,5.46,17.68,68.32,3.62,9.05
